In [1]:
import mlflow

mlflow.set_tracking_uri("file:./mlruns")

In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location=('file:c:/Users/Soham/Documents/youtube comment '
 'analyzer/mlruns/901947899065852447'), creation_time=1778952120616, experiment_id='901947899065852447', last_update_time=1778952120616, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna


In [4]:
df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [5]:
# =========================================================
# IMPORTS
# =========================================================
import numpy as np
import optuna
import mlflow

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.ensemble import (
    RandomForestClassifier
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# RANDOM SEED
# =========================================================
np.random.seed(42)

# =========================================================
# REMOVE NaN TARGETS
# =========================================================
df = df.dropna(
    subset=['category']
)

# =========================================================
# TRAIN TEST SPLIT
# =========================================================
X_train_text, X_test_text, y_train, y_test = train_test_split(

    df['clean_comment'],
    df['category'],

    test_size=0.2,

    random_state=42,

    stratify=df['category']
)

# =========================================================
# TF-IDF SETTINGS
# =========================================================
ngram_range = (1, 3)

max_features = 5000

# =========================================================
# CROSS VALIDATION
# =========================================================
cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

# =========================================================
# OPTUNA OBJECTIVE FUNCTION
# =========================================================
def objective_rf(trial):

    params = {

        "n_estimators": trial.suggest_int(
            'n_estimators',
            200,
            1000,
            step=100
        ),

        "max_depth": trial.suggest_int(
            'max_depth',
            10,
            100
        ),

        "min_samples_split": trial.suggest_int(
            'min_samples_split',
            2,
            10
        ),

        "min_samples_leaf": trial.suggest_int(
            'min_samples_leaf',
            1,
            4
        ),

        "max_features": trial.suggest_categorical(
            'max_features',
            ['sqrt', 'log2', 0.5, 0.75]
        ),

        "class_weight": "balanced",

        "random_state": 42,

        "n_jobs": -1
    }

    # =====================================================
    # PIPELINE
    # =====================================================
    pipeline = Pipeline([

        (
            "tfidf",

            TfidfVectorizer(

                ngram_range=ngram_range,

                max_features=max_features,

                min_df=2,

                max_df=0.95
            )
        ),

        (
            "model",

            RandomForestClassifier(
                **params
            )
        )
    ])

    # =====================================================
    # CROSS VALIDATION
    # =====================================================
    scores = cross_val_score(

        pipeline,

        X_train_text,

        y_train,

        cv=cv,

        scoring='f1_macro',

        n_jobs=-1
    )

    return np.mean(scores)

# =========================================================
# RUN OPTUNA
# =========================================================
study = optuna.create_study(
    direction="maximize"
)

study.optimize(

    objective_rf,

    n_trials=30
)

# =========================================================
# BEST PARAMETERS
# =========================================================
print("=" * 60)

print("BEST PARAMETERS")

print(study.best_params)

print("=" * 60)

# =========================================================
# FINAL PIPELINE
# =========================================================
best_pipeline = Pipeline([

    (
        "tfidf",

        TfidfVectorizer(

            ngram_range=ngram_range,

            max_features=max_features,

            min_df=2,

            max_df=0.95
        )
    ),

    (
        "model",

        RandomForestClassifier(

            **study.best_params,

            class_weight='balanced',

            random_state=42,

            n_jobs=-1
        )
    )
])

# =========================================================
# TRAIN FINAL MODEL
# =========================================================
best_pipeline.fit(

    X_train_text,

    y_train
)

# =========================================================
# PREDICTIONS
# =========================================================
y_pred = best_pipeline.predict(
    X_test_text
)

# =========================================================
# METRICS
# =========================================================
accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(

    y_test,

    y_pred,

    average='macro'
)

weighted_f1 = f1_score(

    y_test,

    y_pred,

    average='weighted'
)

# =========================================================
# RESULTS
# =========================================================
print(f"Accuracy    : {accuracy:.4f}")

print(f"Macro F1    : {macro_f1:.4f}")

print(f"Weighted F1 : {weighted_f1:.4f}")

# =========================================================
# CLASSIFICATION REPORT
# =========================================================
print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# =========================================================
# CONFUSION MATRIX
# =========================================================
conf_matrix = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(8,6))

sns.heatmap(

    conf_matrix,

    annot=True,

    fmt='d',

    cmap='Blues'
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Random Forest Confusion Matrix")

plt.show()

# =========================================================
# MLFLOW LOGGING
# =========================================================
with mlflow.start_run():

    # -----------------------------------------------------
    # TAGS
    # -----------------------------------------------------
    mlflow.set_tag(
        "model",
        "RandomForest"
    )

    mlflow.set_tag(
        "vectorizer",
        "TF-IDF"
    )

    # -----------------------------------------------------
    # PARAMETERS
    # -----------------------------------------------------
    mlflow.log_param(
        "ngram_range",
        ngram_range
    )

    mlflow.log_param(
        "max_features",
        max_features
    )

    mlflow.log_params(
        study.best_params
    )

    # -----------------------------------------------------
    # METRICS
    # -----------------------------------------------------
    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    mlflow.log_metric(
        "macro_f1",
        macro_f1
    )

    mlflow.log_metric(
        "weighted_f1",
        weighted_f1
    )

    # -----------------------------------------------------
    # CLASSIFICATION REPORT
    # -----------------------------------------------------
    report = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in report.items():

        if isinstance(metrics, dict):

            for metric_name, metric_value in metrics.items():

                mlflow.log_metric(
                    f"{label}_{metric_name}",
                    metric_value
                )

    # -----------------------------------------------------
    # SAVE MODEL
    # -----------------------------------------------------
    mlflow.sklearn.log_model(

        best_pipeline,

        "random_forest_pipeline"
    )

[I 2026-05-17 22:54:47,923] A new study created in memory with name: no-name-ac137410-dacf-4208-bbbf-e3bc492536fb
[I 2026-05-17 22:55:23,048] Trial 0 finished with value: 0.705763512290593 and parameters: {'n_estimators': 1000, 'max_depth': 82, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.705763512290593.
[I 2026-05-17 23:14:13,319] Trial 1 finished with value: 0.7212765084134374 and parameters: {'n_estimators': 900, 'max_depth': 92, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 0.5}. Best is trial 1 with value: 0.7212765084134374.
[I 2026-05-17 23:14:25,812] Trial 2 finished with value: 0.6928290171366044 and parameters: {'n_estimators': 400, 'max_depth': 71, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 1 with value: 0.7212765084134374.
[I 2026-05-17 23:22:16,548] Trial 3 finished with value: 0.6492345210588675 and parameters: {'n_estimators': 600, 'max_depth': 31, 'min_

KeyboardInterrupt: 